# 103 - Fixed-Sampling DFT Propagation

The 101 notebook used fast fourier transform (FFT) propagation, which is
the colloquial method to do this propagation in most literature and many
software packages.  The FFT propagation is fast, but has fixed and implicit
sampling:

$$ dx_\text{PSF} = \frac{\lambda F\#}{Q/2}\quad\text{(microns)} $$

where $Q$ is the padding factor.  If you want to make dx more fine, you must
zero pad the pupil.  $dx_\text{PSF}$ is also chromatic when using an FFT,
requiring an interpolation step when doing broadband propagations, and if you
only need a region near the core of the PSF, you have to compute the high
frequency content and then throw it away.

This can be avoided by using either the matrix discrete fourier transform
(matrix DFT, or MDFT) or chirp Z transform (CZT).  Both are implemented in
prysm.  To understand the MDFT, recall that the Fourier spectrum is the dot
product between sines/cosines and the signal:

```python
sx = sin(2*pi*x*f)  # sine x
cx = cos(2*pi*x*f)  # cos x
csf = dot(sx, data) # coefficient of sine x
ccf = dot(cx, data) #                cos  x
```

The two can be computed simultaneously using Euler's identity:
`exp(i*x) = sin(c) + i*cos(x)` and the dot products can be batched
by performing a matrix multiply.  So, the MDFT uses two matrix multiplies
to perform the 2D DFT.  The CZT uses an FFT, transformation, then inverse
FFT to perform a DFT with arbitrary sampling.  This makes the CZT a bit
more than 2x the computational cost as an FFT of the natural size, which
is much faster than heavy zero padding.  For example, if your pupil has
256 samples across and you want to compute a propagation at Q=8 needing
only the middle 64 pixels of the output, the CZT will do a 320-element FFT
instead of a 2048-element FFT, which is much faster.

The user need not have a deep understanding of the mathematics of these
methods; the DFT propagation API allows any of them to be selected with a
simple keyword.  As a user, you can benchmark any of them and select the most
optimal for your given propagation and hardware.

We'll demonstrate MDFT- and CZT-based propagations in this notebook,
using the same system as was used in the 101 notebook.  We begin with the usual
imports and set up our simple reference F/10, 100mm focal length system at HeNe:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle
from prysm.propagation import Wavefront
from prysm.wavelengths import HeNe

# a 10 mm diameter, f/10 lens (100 mm efl) at the HeNe wavelength
EPD = 10.0          # entrance pupil diameter, mm
EFL = 100.0         # focal length, mm
FNO = EFL / EPD     # f/10
WVL = HeNe          # 0.6328 um

xi, eta = make_xy_grid(256, diameter=EPD)
r, t = cart_to_polar(xi, eta)
dx = xi[0, 1] - xi[0, 0]            # pupil sample spacing, mm
A = circle(EPD / 2, r)             # the clear circular aperture
airy_radius = 1.22 * WVL * FNO     # first dark ring, microns

## FFT sampling

To motivate things, we'll start by computing the PSF via FFT:

Here is the ideal PSF at two padding factors.  `Q=2` is the minimum for an
unaliased incoherent PSF; `Q=8` is four times finer.  Note the printed sample
spacings: the only way to change them with the FFT is to change `Q`.

In [ ]:
wf = Wavefront.from_amp_and_phase(A, None, WVL, dx)
psf_q2 = wf.focus(EFL, Q=2).intensity
psf_q8 = wf.focus(EFL, Q=8).intensity
print(f'Q=2 focal dx = {psf_q2.dx:.3f} um   ({psf_q2.data.shape[0]} samples)')
print(f'Q=8 focal dx = {psf_q8.dx:.3f} um   ({psf_q8.data.shape[0]} samples)')
print(f'lambda*F#    = {WVL*FNO:.3f} um')
psf_q8.plot2d(xlim=12*airy_radius, power=1/4);

## MDFT and CZT

When setting up a DFT propagation, your code is broken into two steps:

- set up the cached propagator
- run the propagation

This is similar to in the polynomials module, computing the basis functions
and then using `sum_of_2d_modes` many times with the same dense basis cube.

We'll build a grid that is 128 samples across and equivalent to the Q=8 sampling:

In [ ]:
dxfine = WVL * FNO / 8
Nfine = 128

ex_dft = wf.prepare_executor(EFL, dxfine, Nfine, kind='mdft')
ex_czt = wf.prepare_executor(EFL, dxfine, Nfine, kind='czt')

psf_dft = wf.focus_dft(ex_dft).intensity
psf_czt = wf.focus_dft(ex_czt).intensity

print(f'DFT focal dx = {psf_dft.dx:.3f} um   ({psf_dft.data.shape[0]} samples)')
psf_dft.plot2d(xlim=12*airy_radius, power=1/4);

The grid is smaller than the region that was plotted, but the data within it is
the same.  Both MDFT and CZT allow arbitrary grids to be computed.  We can
show in 1D that the data is the same:

In [ ]:
psf_q8n = psf_q8.copy();
cq, vq = psf_q8n.slices().x
cd, vd = psf_dft.slices().x

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cq, vq, 'o', ms=3, label='FFT (Q=2)')
ax.plot(cd, vd, '-', label='matrix DFT (4x fine)')
ax.set(yscale='log', ylim=(1e-4, 800), xlim=(0, airy_radius*5),
       xlabel='x [um]', ylabel='normalized intensity',
       title='FFT and matrix DFT sample the same transform')
ax.legend()

## Round trip: focal plane back to pupil

Many models, for example coronagraphs, need to propagate `pupil=>focus=>pupil`;
this can be done using `unfocus_dft`.  In general, this has both aliasing
implications as well as conservation of energy (or lack thereof) implications.

A small amount of energy can be created or destroyed by performing a pair of
propagations when the PSF is not bandlimited in the limited area computed.
This artifact is present in this example, where 0.4% energy appears to be
created due to aliasing.  Ringing can also be seen in the interior of the
pupil after returning to the pupil; this is due to the inherent low-pass filter
applied by back-propagating only a limited portion of the PSF.

In [ ]:
ex_dft_wide = wf.prepare_executor(EFL, WVL*FNO/2, 256, kind='mdft')
field_focal = wf.focus_dft(ex_dft_wide)
field_pupil = field_focal.unfocus_dft(ex_dft_wide)

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
axs[0].imshow(np.abs(wf.data), extent=[-EPD/2, EPD/2, -EPD/2, EPD/2], cmap='gray', clim=(0,1))
axs[0].set(title=f'input pupil amplitude\nsum {np.abs(wf.data).sum():.0f}')
axs[1].imshow(np.abs(field_pupil.data), extent=[-EPD/2, EPD/2, -EPD/2, EPD/2], cmap='gray', clim=(0,1))
axs[1].set(title=f'after focus_dft + unfocus_dft\nsum {np.abs(field_pupil.data).sum():.0f}')
fig.tight_layout()

## Wrap-up

DFT or fixed sampling propagations are performed in two steps:

- prepare_executor
- focus_dft / unfocus_dft

This is in contrast to the FFT API, which is one line.  The DFT API provides
exact control over sampling, and is used in polychromatic propagation (201),
coronagraph modeling (301, 302).  In the majority of applications, you will
likely prefer using the dft approach to FFT for performance, or convenience
of using the output.